# DPF — Distributed Point Function (BGI16)

ELL_IN ∈ [13, 31], ELL_OUT ∈ [2, 6].  Star topology: one Dealer,
two Evaluators.  Based on Boyle, Gilboa, and Ishai (CCS 2016).

- ``DpfDealer(ell_in, ell_out)`` — generates two keys
- ``DpfEvaluator(ell_in, ell_out, party)`` — evaluates one key
  (*party* is 0 or 1)

Both are **instance methods** — construct the object first, then call
``gen`` / ``eval`` on it.


In [ ]:
import mpmt

dpf_d = mpmt.DpfDealer(ell_in=20, ell_out=4)
dpf_e0 = mpmt.DpfEvaluator(ell_in=20, ell_out=4, party=0)
dpf_e1 = mpmt.DpfEvaluator(ell_in=20, ell_out=4, party=1)
print(f"DPF: in={dpf_d.ell_in}, out={dpf_d.ell_out}")


## Dealer: gen

``dpf_dealer.gen(alpha, beta)`` generates two DPF keys for a point
function f(x) = beta if x == alpha, else 0.  Returns a JSON string
(key).  Send one key to each evaluator over a channel.

*alpha* is the secret point (in Z_{2^{ell_in}}).  *beta* is the
output value (in Z_{2^{ell_out}}).


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

key_e0 = dpf_d.gen(alpha=42, beta=1)
# send key_e0 to Evaluator 0
# send key_e1 to Evaluator 1


## Evaluator: eval (full domain)

``evaluator.eval(key_json, buf, cores=1)`` evaluates the DPF over the
entire domain [0, 2^{ell_in}).  *buf* is a pre-allocated ``RvectorPack``
of ``bf_size`` elements.  Results are written into *buf*.

The two evaluators' results are 2-of-2 additive shares of the point
function: ``eval0[i] + eval1[i] = f(i)``.


In [ ]:
# This cell requires multi-party setup and will not run in a notebook

buf = mpmt.RvectorPack(ell=4)(bf_size)
dpf_e0.eval(key_json=key_e0, buf=buf, cores=1)


## Evaluator: eval_range

``evaluator.eval_range(...)`` evaluates over a sub-range.  Used when
the full domain is too large for a single buffer.


## GenBF Protocol Usage

In the query protocol, the DPF is used to generate the query-side
Bloom filter without revealing the hash indices:

1. **Leader** (acting as S3 / Dealer) generates DPF keys for the
   **blinded** index ``idx_L + idx_A``, where ``idx_L`` is the ADD2
   hash share held by the Leader and ``idx_A`` is a random offset.
   Keys are sent to HelperA and HelperB.

2. **HelperA** and **HelperB** evaluate their keys over [0, bf_size).
   They each apply a **cyclic shift** by their RSS3 share of the
   blinding offset, producing 2-of-2 additive shares of the query BF.

3. **crng** + **reshare**: the 2-of-2 shares are converted to Rep3
   (2-of-3) via correlated randomness and a reshare round, so the
   three servers can compute the dot product.


## Channel Topology

DPF uses a **star** topology (not a ring):
```
        Dealer
       /      \
  Eval0      Eval1
```

Channels are built with ``_build_dpf_channels_dealer`` (listens on two
ports) and ``_build_dpf_channels_evaluator`` (connects to one port).


## Properties

- ``dealer.ell_in``, ``dealer.ell_out`` — ring sizes
- ``evaluator.ell_in``, ``evaluator.ell_out``, ``evaluator.party``
